In [10]:
from transformers import AutoModelForCausalLM,AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1332.76it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
#1、第一步：对数据集进行处理，处理成trl所需要的language modeling类型的对话格式
from datasets import load_dataset
# 1.1、将两个数据集文件，分别加载到dataset dict当中的train 和test分片中
dataset_dict = load_dataset("json",data_files = {"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 500
    })
})

In [12]:
from typing import List,Dict
# 1.2 、定义数据处理的 mapping function
def convert_function(examples:Dict[str, List]):
    """
    数据集分批次调用该方法，修改数据格式
    """
    conversation_list:List[List[Dict]] = examples["conversation"]
    message_list = []
    for data in conversation_list:
        # data是单条样本
        coversation_dict = data[0]
        current_data_message_list = [
            {"role":"user","content":coversation_dict["human"]},
            {"role":"assistant","content":coversation_dict["assistant"]}
        ]
        message_list.append(current_data_message_list)
    
    return {"messages":message_list}

# 1.3、调用dataset_dict.map方法，传入mapping function，对数据进行格式化
mapped_dataset_dict = dataset_dict.map(convert_function,batched=True,remove_columns = ['conversation_id', 'category', 'conversation', 'dataset'])
mapped_dataset_dict

Map: 100%|██████████| 500/500 [00:00<00:00, 25935.91 examples/s]


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 500
    })
})

In [9]:
mapped_dataset_dict["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [ ]:
# 2、构造SFTTrainer实例
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
# 2.1、构造训练时的参数SFTConfig
SFTConfig(
    
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = mapped_dataset_dict["train"],
    eval_dataset = mapped_dataset_dict["test"],
    args = sft_config
)